## Hybrid Search Langchain

In [1]:
api_key = "pcsk_6bz1zg_KN5HiDoG9iEHEMuiuhnGmMNmHQfLW6F44VontMqMWjtGtn9y3TQE6ANKaJxGUPc"

In [2]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

/var/folders/h_/3_y8qn2x5t5_7mw2r32c4dp40000gn/T/ipykernel_46154/1338964493.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import PineconeHybridSearchRetriever


In [3]:
import os
from pinecone import Pinecone, ServerlessSpec

## 1. Set up Pinecone client and create index (only need to create once)
pc = Pinecone(api_key=api_key)

index_name = "hybrid-search-demo"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,  # must match your embedding model's output size (all-MiniLM-L6-v2 = 384)
        metric="dotproduct",  # hybrid search REQUIRES dotproduct metric
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [4]:
index

Index(host='https://hybrid-search-demo-2wtpioj.svc.aped-4627-b74a.pinecone.io')

In [5]:
#Vector Embedding and Sparse Matrix

import os
from dotenv import load_dotenv

load_dotenv()

from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


/Users/venu/Documents/AI/LangChain/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9101.92it/s]


In [7]:
from pinecone_text.sparse import BM25Encoder
## 3. Set up sparse encoder (keyword search) — BM25
bm25_encoder = BM25Encoder().default()
bm25_encoder

In [13]:
## 5. Add documents (only need to do this once — persists in Pinecone)
sentences = [
    "The Eiffel Tower is located in Paris, France.",
    "Python is a popular programming language for data science.",
    "LangChain helps developers build LLM-powered applications.",
    "The Great Wall of China is visible from low Earth orbit.",
    "Pinecone is a vector database used for similarity search.",
]

bm25_encoder.fit(sentences)

#store to json file
bm25_encoder.dump("bm25_values.json")

bm25_encoder = BM25Encoder().load("bm25_values.json")

100%|██████████| 5/5 [00:00<00:00, 9981.68it/s]


In [23]:
## 4. Create the hybrid retriever

# Newer pinecone client versions made Index.upsert() keyword-only, but
# langchain_community's PineconeHybridSearchRetriever calls it positionally.
# Wrap the index so positional upsert calls are forwarded as keyword args.
class _KeywordOnlyIndexWrapper:
    def __init__(self, wrapped_index):
        self._wrapped_index = wrapped_index

    def upsert(self, vectors, namespace=None, **kwargs):
        return self._wrapped_index.upsert(vectors=vectors, namespace=namespace, **kwargs)

    def __getattr__(self, name):
        return getattr(self._wrapped_index, name)

retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25_encoder,
    index=_KeywordOnlyIndexWrapper(index),
    alpha=0.5,  # 0 = pure keyword search, 1 = pure semantic search, 0.5 = balanced
    top_k=3
)

In [24]:
retriever.add_texts(sentences)

## 6. Query it!
results = retriever.invoke("What is used to build applications with LLMs?")

for i, doc in enumerate(results):
    print(f"{i+1}. {doc.page_content}")
    

100%|██████████| 1/1 [00:01<00:00,  1.42s/it]


1. LangChain helps developers build LLM-powered applications.
2. Pinecone is a vector database used for similarity search.
3. Python is a popular programming language for data science.


In [16]:
results

[]